# Week 63, Day 5: Latent variable models of spiking populations

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w063/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
g0 = np.random.default_rng(635)
T, N = 300, 40
t = np.arange(T)
z_true = np.sin(2 * np.pi * t / 120) + 0.6 * np.sin(2 * np.pi * t / 47 + 1.0)
z_true = (z_true - z_true.mean()) / z_true.std()
c = g0.normal(0, 0.6, N)
d = np.log(0.5) * np.ones(N)                           # about half a spike per bin at z = 0
rates = np.exp(np.outer(z_true, c) + d)
X = g0.poisson(rates)
print(f"{X.sum()} spikes; mean {X.mean():.2f} per neuron per bin; {np.mean(X == 0):.0%} of bins empty")

### Exercise: Per-bin variational inference

Complete the Newton updates for m (all bins at once): the gradient of L_t with respect to m_t is sum_n c_n (x_tn - lam_tn) - m_t, and the second derivative is -sum_n c_n^2 lam_tn - 1, where lam_tn is exp(c_n m_t + d_n + c_n^2 v_t / 2). Then update v from its fixed point 1 / (sum_n c_n^2 lam_tn + 1).

In [ ]:
m = np.zeros(T); v = np.ones(T)
for it in range(50):
    lam = np.exp(np.outer(m, c) + d + np.outer(v, c ** 2) / 2)
    grad = ...
    hess = -(lam * c ** 2).sum(1) - 1
    m = m - grad / hess
    lam = np.exp(np.outer(m, c) + d + np.outer(v, c ** 2) / 2)
    v = ...

r_vi = np.corrcoef(m, z_true)[0, 1]
proj = np.sqrt(X) @ c                                  # project square-root counts on the loadings
r_sqrt = abs(np.corrcoef(proj, z_true)[0, 1])
Xc = np.sqrt(X) - np.sqrt(X).mean(0)
pc1 = Xc @ np.linalg.svd(Xc, full_matrices=False)[2][0]
r_pca = abs(np.corrcoef(pc1, z_true)[0, 1])
print(f"correlation with the true latent: Poisson VI {r_vi:.3f}, square-root projection {r_sqrt:.3f}, first PC {r_pca:.3f}")
print(f"posterior sd ranges from {np.sqrt(v).min():.2f} to {np.sqrt(v).max():.2f}")

In [ ]:
import numpy as np
def test_beats():
    "the Poisson variational posterior mean tracks the latent better than the square-root projection"
    assert r_vi > r_sqrt
def test_var():
    "posterior variances are smaller where the latent drives high rates (more spikes, more information)"
    assert np.corrcoef(v, rates.sum(1))[0, 1] < -0.3

test_beats()
test_var()
print("All checks passed")

### Exercise: An autoregressive prior

Build the AR(1) prior precision Q for a = 0.97 (tridiagonal: 1 / (1 - a^2) times [1, 1 + a^2, ..., 1 + a^2, 1] on the diagonal and -a / (1 - a^2) beside it). Then run Newton's method on the log posterior of the whole trajectory: gradient c-weighted residuals minus Q z, Hessian minus (diag(sum_n c_n^2 lam_tn) + Q).

In [ ]:
a = 0.97
main = np.r_[1.0, np.full(T - 2, 1 + a ** 2), 1.0] / (1 - a ** 2)
off = np.full(T - 1, -a / (1 - a ** 2))
Q = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)

zs = np.zeros(T)
for it in range(30):
    lam = np.exp(np.outer(zs, c) + d)
    grad = ...
    H = np.diag((lam * c ** 2).sum(1)) + Q
    zs = zs + np.linalg.solve(H, grad)
r_smooth = np.corrcoef(zs, z_true)[0, 1]
sd_post = np.sqrt(np.diag(np.linalg.inv(H)))
print(f"correlation with the true latent: independent bins {r_vi:.3f}, with the smoothness prior {r_smooth:.3f}")
plt.figure(figsize=(9, 3)); plt.plot(z_true, "k", lw=2, label="true")
plt.plot(m, color="C1", alpha=0.6, lw=1, label="independent bins")
plt.plot(zs, color="C0", lw=1.5, label="smoothness prior")
plt.fill_between(t, zs - 2 * sd_post, zs + 2 * sd_post, color="C0", alpha=0.2); plt.legend(); plt.xlabel("bin"); plt.show()

In [ ]:
import numpy as np
def test_better():
    "the smoothness prior raises the correlation with the true latent by at least 0.05 and above 0.9"
    assert r_smooth > r_vi + 0.05 and r_smooth > 0.9

test_better()
print("All checks passed")